In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Now you can access the Radiography folder
radiography_path = '/content/drive/My Drive/Radiography'

# For example, to list all files in the Radiography folder
import os

files = os.listdir(radiography_path)
print(files)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['gantt_chart.png', 'project plan.py', 'enhanced_gantt_chart_with_duration.png', 'model.keras', 'final_model.h5', 'test.py', 'v3.keras', '__pycache__', '.git', 'images', 'report', '.ipynb_checkpoints', 'ipynb.ipynb', 'main.py', 'model_development.py', 'preprocessing.py', 'data_exploration.py']


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import cv2
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns


def plot_image_distribution(preprocessed_images):
    # Plot the distribution of pixel values in the preprocessed images
    pixel_values = preprocessed_images.flatten()
    plt.hist(pixel_values, bins=50, density=True)
    plt.xlabel("Pixel Values")
    plt.ylabel("Frequency")
    plt.title("Distribution of Pixel Values")
    plt.show()


def plot_report_lengths(preprocessed_reports):
    # Plot the distribution of report lengths
    report_lengths = [len(report) for report in preprocessed_reports]
    plt.hist(report_lengths, bins=20)
    plt.xlabel("Report Length")
    plt.ylabel("Frequency")
    plt.title("Distribution of Report Lengths")
    plt.show()


def analyze_report_vocabulary(preprocessed_reports):
    # Analyze the vocabulary used in the reports
    all_tokens = [token for report in preprocessed_reports for token in report]
    token_counts = Counter(all_tokens)
    print("Total Vocabulary Size:", len(token_counts))
    print("Top 10 Most Common Tokens:", token_counts.most_common(10))


# image qulaity anlysis
def calculate_image_sharpness(image):
    image = np.array(image * 255, dtype=np.uint8)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian


def check_image_quality(preprocessed_images):
    sharpness_scores = [calculate_image_sharpness(image) for image in preprocessed_images]
    plt.hist(sharpness_scores, bins=50)
    plt.xlabel('Sharpness Score')
    plt.ylabel('Frequency')
    plt.title('Distribution of Image Sharpness Scores')
    plt.show()

    print("Mean Sharpness:", np.mean(sharpness_scores))
    print("Median Sharpness:", np.median(sharpness_scores))
    print("Standard Deviation of Sharpness:", np.std(sharpness_scores))


# class distribution
def plot_class_distribution(image_report_map):
    report_ids = list(image_report_map.values())
    report_counts = Counter(report_ids)
    plt.hist(report_counts.values(), bins=20)
    plt.xlabel("Number of Images")
    plt.ylabel("Frequency")
    plt.title("Distribution of Number of Images per Report")
    plt.show()

    print("Number of Unique Reports:", len(report_counts))
    print("Top 10 Reports with Most Images:")
    print(report_counts.most_common(10))

# plot the top 10 reports with most images and reports with least images

#report simillaritiws  analysis
def plot_report_similarity(preprocessed_reports):
    tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x)
    tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_reports)
    cosine_sim = cosine_similarity(tfidf_matrix)

    plt.figure(figsize=(10, 10))
    sns.heatmap(cosine_sim, cmap='coolwarm')
    plt.title('Report Similarity Matrix')
    plt.show()

    similar_reports = []
    for i in range(len(cosine_sim)):
        for j in range(i + 1, len(cosine_sim)):
            if cosine_sim[i][j] > 0.9:
                similar_reports.append((i, j))
    print("Number of Similar Reports:", len(similar_reports))
    print("Similar Report Pairs:")
    print(similar_reports)


def explore_data(preprocessed_images, preprocessed_reports, image_report_map):
    print("Data Exploration")
    print("-----------------")

    print("Number of Images:", len(preprocessed_images))
    print("Image Shape:", preprocessed_images[0].shape)
    plot_image_distribution(preprocessed_images)

    print("\nNumber of Reports:", len(preprocessed_reports))
    plot_report_lengths(preprocessed_reports)

    print("\nVocabulary Analysis:")
    analyze_report_vocabulary(preprocessed_reports)

    print("\nImage Quality Analysis:")
    check_image_quality(preprocessed_images)

    print("\nClass Distribution:")
    plot_class_distribution(image_report_map)

    print("\nReport Similarity Analysis:")
    plot_report_similarity(preprocessed_reports)




In [ ]:
import os
import xml.etree.ElementTree as ET
import nltk
from matplotlib import pyplot as plt
from nltk.tokenize import word_tokenize
from PIL import Image
import numpy as np

#from data_exploration import calculate_image_sharpness


# Function to preprocess reports
def preprocess_reports(report_dir):
    preprocessed_reports = []
    image_report_map = {}
    report_filenames = []
    for filename in os.listdir(report_dir):
        if filename.endswith(".xml"):
            report_path = os.path.join(report_dir, filename)
            tree = ET.parse(report_path)
            root = tree.getroot()

            findings = ""
            impression = ""
            for abstract_text in root.findall(".//AbstractText"):
                if abstract_text.get("Label") == "FINDINGS":
                    findings = abstract_text.text if abstract_text.text else ""
                elif abstract_text.get("Label") == "IMPRESSION":
                    impression = abstract_text.text if abstract_text.text else ""

            findings_tokens = word_tokenize(findings) if findings else []
            impression_tokens = word_tokenize(impression) if impression else []
            preprocessed_reports.append((findings_tokens, impression_tokens))
            report_filenames.append(filename)

            # Extract image IDs from the XML
            image_ids = []
            for parent_image in root.findall(".//parentImage"):
                image_id = parent_image.get("id")
                image_ids.append(image_id)

            # Map report filename to image filenames
            report_id = filename.split(".")[0]
            for image_id in image_ids:
                image_report_map[image_id] = report_id

    return preprocessed_reports, image_report_map, report_filenames

# Function to preprocess images
def preprocess_images(image_dir, target_size=(224, 224)):
    preprocessed_images = []
    image_filenames = []

    for filename in os.listdir(image_dir):
        if filename.endswith(".png"):
            image_path = os.path.join(image_dir, filename)
            image = Image.open(image_path)
            image = image.resize(target_size)
            image = np.array(image) / 255.0
            preprocessed_images.append(image)
            image_filenames.append(filename)

    preprocessed_images = np.array(preprocessed_images)
    return preprocessed_images, image_filenames

# check for the description of the reports


In [ ]:
from keras.src.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Concatenate, GlobalAveragePooling2D, LSTM, Embedding, GlobalAveragePooling1D, \
    Reshape, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.applications import DenseNet121, InceptionV3, VGG16, ResNet50
import numpy as np
import tensorflow as tf


def split_data(preprocessed_images, preprocessed_reports, test_size=0.2, random_state=42):
    train_images, test_images, train_reports, test_reports = train_test_split(
        preprocessed_images, preprocessed_reports, test_size=test_size, random_state=random_state)
    return train_images, test_images, train_reports, test_reports


def create_cnn_model(input_shape):
    #base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    # using inceptionv3 model
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)  # Adding dropout
    x = Dense(128, activation='relu')(x)
    cnn_model = Model(inputs=base_model.input, outputs=x)
    return cnn_model


def create_rnn_model(max_length, vocab_size):
    input_ids = Input(shape=(max_length,), dtype='int32', name='input_ids')
    x = Embedding(input_dim=vocab_size, output_dim=128)(input_ids)
    x = LSTM(128, return_sequences=True)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.5)(x)  # Adding dropout
    x = Dense(128, activation='relu')(x)
    rnn_model = Model(inputs=input_ids, outputs=x)
    return rnn_model


def create_model(cnn_model, rnn_model, vocab_size, max_length):
    image_input = Input(shape=cnn_model.input_shape[1:])
    image_features = cnn_model(image_input)

    input_ids = Input(shape=(max_length,), dtype='int32', name='input_ids')
    report_features = rnn_model(input_ids)

    combined_features = Concatenate()([image_features, report_features])
    dense_layer = Dense(256, activation='relu')(combined_features)

    # Adjust the output layer to produce the correct shape directly
    output_layer = Dense(max_length * vocab_size, activation='softmax')(dense_layer)
    output_layer = Reshape((max_length, vocab_size))(output_layer)

    findings_output = Dense(max_length * vocab_size, activation='softmax')(dense_layer)
    findings_output = Reshape((max_length, vocab_size), name='findings_output')(findings_output)

    impression_output = Dense(max_length * vocab_size, activation='softmax')(dense_layer)
    impression_output = Reshape((max_length, vocab_size), name='impression_output')(impression_output)

    model = Model(inputs=[image_input, input_ids], outputs=[findings_output, impression_output])
    return model


def train_model(model, train_images, train_report_ids, epochs, batch_size, train_findings_labels, train_impression_labels):
    train_images = np.array(train_images)
    train_report_ids = np.array(train_report_ids)
    train_findings_labels = np.array(train_findings_labels)
    train_impression_labels = np.array(train_impression_labels)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy', metrics=[['accuracy'], ['accuracy']])
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    model_checkpoint = ModelCheckpoint('v3.keras', save_best_only=True)

    print(f"Shape of train_images: {train_images.shape}")
    print(f"Shape of train_report_ids: {train_report_ids.shape}")
    print(f"Shape of train_findings_labels: {train_findings_labels.shape}")
    print(f"Shape of train_impression_labels: {train_impression_labels.shape}")

    model.fit([train_images, train_report_ids], [train_findings_labels, train_impression_labels],
              epochs=int(epochs), batch_size=batch_size, validation_split=0.1,
              callbacks=[early_stopping, model_checkpoint])

    return model

def evaluate_model(model, test_images, test_report_ids, test_findings_labels, test_impression_labels):
    loss, findings_loss, impression_loss, findings_acc, impression_acc = model.evaluate([test_images, test_report_ids],
                                                                                        [test_findings_labels,
                                                                                         test_impression_labels])
    print("Test Loss:", loss)
    print("Findings Loss:", findings_loss)
    print("Impression Loss:", impression_loss)
    print("Findings Accuracy:", findings_acc)
    print("Impression Accuracy:", impression_acc)


In [ ]:
import os
import xml.etree.ElementTree as ET
import nltk
from matplotlib import pyplot as plt
from nltk.tokenize import word_tokenize
from PIL import Image
import numpy as np
from tf_keras.src.callbacks import EarlyStopping, ModelCheckpoint
#from model_development import create_model, create_cnn_model, create_rnn_model, split_data, train_model, evaluate_model
#from preprocessing import preprocess_reports, preprocess_images
#from data_exploration import explore_data
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


def prepare_labels(reports, tokenizer, max_length, num_classes):
    findings_labels = []
    impression_labels = []
    for report in reports:
        if "IMPRESSION" in report:
            findings, impression = report.split("IMPRESSION")[0], "IMPRESSION" + report.split("IMPRESSION")[1]
        else:
            findings, impression = report, ""
        findings_encoding = tokenizer.texts_to_sequences([findings])[0]
        impression_encoding = tokenizer.texts_to_sequences([impression])[0]
        findings_label = pad_sequences([findings_encoding], maxlen=max_length)[0]
        impression_label = pad_sequences([impression_encoding], maxlen=max_length)[0]
        findings_labels.append(to_categorical(findings_label, num_classes=num_classes))
        impression_labels.append(to_categorical(impression_label, num_classes=num_classes))
    return np.array(findings_labels), np.array(impression_labels)

def preprocess_images(image_dir, target_size=(224, 224)):
    preprocessed_images = []
    image_filenames = []

    for filename in os.listdir(image_dir):
        if filename.endswith(".png"):
            image_path = os.path.join(image_dir, filename)
            image = Image.open(image_path)
            image = image.resize(target_size)
            image = np.array(image) / 255.0
            preprocessed_images.append(image)
            image_filenames.append(filename)

    preprocessed_images = np.array(preprocessed_images)
    return preprocessed_images, image_filenames

def preprocess_reports(report_dir):
    preprocessed_reports = []
    image_report_map = {}
    report_filenames = []
    for filename in os.listdir(report_dir):
        if filename.endswith(".xml"):
            report_path = os.path.join(report_dir, filename)
            tree = ET.parse(report_path)
            root = tree.getroot()

            findings = ""
            impression = ""
            for abstract_text in root.findall(".//AbstractText"):
                if abstract_text.get("Label") == "FINDINGS":
                    findings = abstract_text.text if abstract_text.text else ""
                elif abstract_text.get("Label") == "IMPRESSION":
                    impression = abstract_text.text if abstract_text.text else ""

            findings_tokens = word_tokenize(findings) if findings else []
            impression_tokens = word_tokenize(impression) if impression else []
            preprocessed_reports.append((findings_tokens, impression_tokens))
            report_filenames.append(filename)

            # Extract image IDs from the XML
            image_ids = []
            for parent_image in root.findall(".//parentImage"):
                image_id = parent_image.get("id")
                image_ids.append(image_id)

            # Map report filename to image filenames
            report_id = filename.split(".")[0]
            for image_id in image_ids:
                image_report_map[image_id] = report_id

    return preprocessed_reports, image_report_map, report_filenames


from keras.src.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Concatenate, GlobalAveragePooling2D, LSTM, Embedding, GlobalAveragePooling1D, \
    Reshape, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.applications import DenseNet121, InceptionV3, VGG16, ResNet50
import numpy as np
import tensorflow as tf


def split_data(preprocessed_images, preprocessed_reports, test_size=0.2, random_state=42):
    train_images, test_images, train_reports, test_reports = train_test_split(
        preprocessed_images, preprocessed_reports, test_size=test_size, random_state=random_state)
    return train_images, test_images, train_reports, test_reports


def create_cnn_model(input_shape):
    #base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    # using inceptionv3 model
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)  # Adding dropout
    x = Dense(128, activation='relu')(x)
    cnn_model = Model(inputs=base_model.input, outputs=x)
    return cnn_model


def create_rnn_model(max_length, vocab_size):
    input_ids = Input(shape=(max_length,), dtype='int32', name='input_ids')
    x = Embedding(input_dim=vocab_size, output_dim=128)(input_ids)
    x = LSTM(128, return_sequences=True)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.5)(x)  # Adding dropout
    x = Dense(128, activation='relu')(x)
    rnn_model = Model(inputs=input_ids, outputs=x)
    return rnn_model


def create_model(cnn_model, rnn_model, vocab_size, max_length):
    image_input = Input(shape=cnn_model.input_shape[1:])
    image_features = cnn_model(image_input)

    input_ids = Input(shape=(max_length,), dtype='int32', name='input_ids')
    report_features = rnn_model(input_ids)

    combined_features = Concatenate()([image_features, report_features])
    dense_layer = Dense(256, activation='relu')(combined_features)

    # Adjust the output layer to produce the correct shape directly
    output_layer = Dense(max_length * vocab_size, activation='softmax')(dense_layer)
    output_layer = Reshape((max_length, vocab_size))(output_layer)

    findings_output = Dense(max_length * vocab_size, activation='softmax')(dense_layer)
    findings_output = Reshape((max_length, vocab_size), name='findings_output')(findings_output)

    impression_output = Dense(max_length * vocab_size, activation='softmax')(dense_layer)
    impression_output = Reshape((max_length, vocab_size), name='impression_output')(impression_output)

    model = Model(inputs=[image_input, input_ids], outputs=[findings_output, impression_output])
    return model


def train_model(model, train_images, train_report_ids, epochs, batch_size, train_findings_labels, train_impression_labels):
    train_images = np.array(train_images)
    train_report_ids = np.array(train_report_ids)
    train_findings_labels = np.array(train_findings_labels)
    train_impression_labels = np.array(train_impression_labels)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy', metrics=[['accuracy'], ['accuracy']])
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    model_checkpoint = ModelCheckpoint('v3.keras', save_best_only=True)

    print(f"Shape of train_images: {train_images.shape}")
    print(f"Shape of train_report_ids: {train_report_ids.shape}")
    print(f"Shape of train_findings_labels: {train_findings_labels.shape}")
    print(f"Shape of train_impression_labels: {train_impression_labels.shape}")

    model.fit([train_images, train_report_ids], [train_findings_labels, train_impression_labels],
              epochs=int(epochs), batch_size=batch_size, validation_split=0.1,
              callbacks=[early_stopping, model_checkpoint])

    return model

def evaluate_model(model, test_images, test_report_ids, test_findings_labels, test_impression_labels):
    results = model.evaluate(
        [np.array(test_images), np.array(test_report_ids)],
        [np.array(test_findings_labels), np.array(test_impression_labels)],
        verbose=1
    )

    if isinstance(results, list):
        if len(results) == 5:
            loss, findings_loss, impression_loss, findings_acc, impression_acc = results
            print("Test Loss:", loss)
            print("Findings Loss:", findings_loss)
            print("Impression Loss:", impression_loss)
            print("Findings Accuracy:", findings_acc)
            print("Impression Accuracy:", impression_acc)
        elif len(results) == 3:
            loss, findings_metric, impression_metric = results
            print("Test Loss:", loss)
            print("Findings Metric:", findings_metric)
            print("Impression Metric:", impression_metric)
        else:
            print("Evaluation Results:", results)
    else:
        print("Test Loss:", results)

    return results


def main():
    # Set the directories for reports and images
    report_dir = "/content/drive/MyDrive/Radiography/report"
    image_dir = "/content/drive/MyDrive/Radiography/images"
    import nltk
    nltk.download('punkt')
    # Preprocess reports
    preprocessed_reports, image_report_map, report_filenames = preprocess_reports(report_dir)

    # Preprocess images
    preprocessed_images, image_filenames = preprocess_images(image_dir)

    # #Print the preprocessed data
    # print("Preprocessed Reports:")
    # print(preprocessed_reports)
    # print("\nImage-Report Map:")
    # print(image_report_map)
    # print("\nPreprocessed Images:")
    # print(preprocessed_images.shape)
    #
    # # Count of the mapped images - reports
    # print("Number of Mapped Images - Reports:", len(image_report_map))
    #
    # # count of the reports
    # print("Number of Reports:", len(preprocessed_reports))
    #
    # # # Perform data exploration
    # explore_data(preprocessed_images, preprocessed_reports, image_report_map)

    # Set the maximum sequence length and vocabulary size for the transformer model
    max_length = 128
    vocab_size = 5000

    # Create the image-report mapping
    image_report_map = {image_filename.split(".")[0]: report_filename.split(".")[0]
                        for image_filename, report_filename in zip(image_filenames, report_filenames)}
    # Filter reports to ensure each report has a corresponding image
    valid_image_filenames = [fname for fname in image_filenames if fname.split(".")[0] in image_report_map]
    valid_preprocessed_images = [preprocessed_images[i] for i, fname in enumerate(image_filenames) if
                                 fname.split(".")[0] in image_report_map]
    valid_preprocessed_reports = [preprocessed_reports[i] for i, fname in enumerate(image_filenames) if
                                  fname.split(".")[0] in image_report_map]

    # Split the data
    train_images, test_images, train_reports, test_reports = split_data(
        valid_preprocessed_images, valid_preprocessed_reports)

    # Convert train and test reports to list of strings
    train_reports = [' '.join(findings + impression) for findings, impression in train_reports]
    test_reports = [' '.join(findings + impression) for findings, impression in test_reports]

    # Tokenize text data
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(train_reports)
    train_report_ids = pad_sequences(tokenizer.texts_to_sequences(train_reports), maxlen=max_length)
    test_report_ids = pad_sequences(tokenizer.texts_to_sequences(test_reports), maxlen=max_length)

    train_findings_labels, train_impression_labels = prepare_labels(train_reports, tokenizer, max_length, num_classes=len(tokenizer.word_index) + 1)

    # Create models
    cnn_model = create_cnn_model(input_shape=(224, 224, 3))
    rnn_model = create_rnn_model(max_length=max_length, vocab_size=len(tokenizer.word_index) + 1)
    model = create_model(cnn_model, rnn_model, vocab_size=len(tokenizer.word_index) + 1, max_length=max_length)

    # Train the model
    train_model(model, train_images, train_report_ids, epochs=1, batch_size=4,
                train_findings_labels=train_findings_labels, train_impression_labels=train_impression_labels)

    test_findings_labels, test_impression_labels = prepare_labels(test_reports, tokenizer, max_length, num_classes=len(tokenizer.word_index) + 1)

    # Evaluate the model
    evaluate_model(model, test_images, test_report_ids, test_findings_labels, test_impression_labels)

    # Save the model in h5 format
    model.save('final_model.h5')

    # Save the tokenizer as a pickle file
    with open('tokenizer.pickle', 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)


if __name__ == "__main__":
    main()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Shape of train_images: (3164, 224, 224, 3)
Shape of train_report_ids: (3164, 128)
Shape of train_findings_labels: (3164, 128, 1882)
Shape of train_impression_labels: (3164, 128, 1882)
25/25 [==============================] - 6s 229ms/step - loss: 1.7235 - findings_output_loss: 1.7098 - impression_output_loss: 0.0137 - findings_output_accuracy: 0.7132 - impression_output_accuracy: 1.0000
Test Loss: 1.7234783172607422
Findings Loss: 1.7097539901733398
Impression Loss: 0.01372471172362566
Findings Accuracy: 0.7132288813591003
Impression Accuracy: 1.0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 43.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 8.5 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import numpy as np
from PIL import Image
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.models import load_model

# Check if model is present
import os

if not os.path.exists("final_model.h5"):
    st.error("Model not found! Please run the model_development.py script to train the model.")
    st.stop()

if not os.path.exists("tokenizer.pickle"):
    st.error("Tokenizer not found! Please make sure you've saved the tokenizer after training.")
    st.stop()

# Load the keras model
model = load_model("final_model.h5")

# Load the tokenizer
with open('tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

# After loading the tokenizer
st.write("Tokenizer vocabulary size:", len(tokenizer.word_index))
st.write("Sample words from tokenizer:", list(tokenizer.word_index.items())[:10])

st.write("Model summary:")
model.summary(print_fn=lambda x: st.text(x))


def preprocess_image(image):
    # Preprocess the uploaded image
    img = image.resize((224, 224))
    img = np.array(img)
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)
    return img


max_length = 128


def generate_report(image, temperature=1.0):
    preprocessed_image = preprocess_image(image)
    input_ids = np.zeros((1, max_length), dtype='int32')
    findings = ""
    impression = ""

    try:
        findings_preds, impression_preds = model.predict([preprocessed_image, input_ids])

        st.write("Findings predictions shape:", findings_preds.shape)
        st.write("Impression predictions shape:", impression_preds.shape)

        for i in range(max_length):
            findings_probs = findings_preds[0, i, :] / temperature
            impression_probs = impression_preds[0, i, :] / temperature

            findings_probs = np.exp(findings_probs) / np.sum(np.exp(findings_probs))
            impression_probs = np.exp(impression_probs) / np.sum(np.exp(impression_probs))

            findings_pred_word = np.random.choice(len(findings_probs), p=findings_probs)
            impression_pred_word = np.random.choice(len(impression_probs), p=impression_probs)

            #st.write(
            # f"Step {i}: Findings word index: {findings_pred_word}, Impression word index: {impression_pred_word}")

            if findings_pred_word != 0:
                findings_word = tokenizer.index_word.get(findings_pred_word, '')
                findings += findings_word + " "

            if impression_pred_word != 0:
                impression_word = tokenizer.index_word.get(impression_pred_word, '')
                impression += impression_word + " "

            if findings_pred_word == 0 and impression_pred_word == 0:
                break

        st.write("Raw findings:", findings)
        st.write("Raw impression:", impression)

        return findings.strip(), impression.strip()
    except Exception as e:
        st.error(f"An error occurred during report generation: {str(e)}")
        return "", ""


# Title
st.title("Radiology Report Generation")
uploaded_image = st.file_uploader("Upload a chest X-ray image", type=["jpg", "jpeg", "png"])

if uploaded_image is not None:
    image = Image.open(uploaded_image)
    st.image(image, caption='Uploaded X-ray', use_column_width=True)

    temperature = st.slider("Temperature", min_value=0.1, max_value=2.0, value=1.0, step=0.1)
    if st.button("Generate Report"):
        findings, impression = generate_report(image, temperature)
        ...
        ...
        st.header("Findings")
        st.write(findings)
        st.header("Impression")
        st.write(impression)
        st.balloons()
        st.success("Report generated successfully!")


In [ ]:
# prompt: generate code to test

# Test the report generation function with a sample image
image_path = "path/to/sample_image.jpg"
image = Image.open(image_path)
findings, impression = generate_report(image)

# Print the generated findings and impression
print("Findings:", findings)
print("Impression:", impression)

# Assert that the generated findings and impression are not empty strings
assert findings != ""
assert impression != ""


NameError: name 'Image' is not defined